In [1]:
import os
import sys

!pip uninstall -y transformers tokenizers huggingface-hub

!pip install transformers==4.51.0 tokenizers==0.21.4 huggingface-hub==0.36.2

!pip install accelerate bitsandbytes
!pip install pyngrok fastapi uvicorn nest_asyncio

# Bersihkan cache
!rm -rf ~/.cache/huggingface

print("[LOAD] Transformers 4.51.0 terinstall. Silakan restart kernel sebelum melanjutkan ke CELL 2.")

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: huggingface_hub 1.11.0
Uninstalling huggingface_hub-1.11.0:
  Successfully uninstalled huggingface_hub-1.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 34.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 36.5 MB/s eta 0:00:0000:0100:01
[LOAD] Transformers 4.51.0 terinstall. Silakan restart kernel sebelum melanjutkan ke CELL 2.


In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import json
import math
import time
import requests
from dataclasses import dataclass
from datetime import datetime
from typing import List, Optional, Tuple, Dict, Any
from functools import lru_cache

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# -----------------------------------------------------------------------------
# 1. Load Model
# -----------------------------------------------------------------------------
def load_model():
    MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"
    print(f"[LOAD] Loading {MODEL_ID} with 4-bit...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    model.eval()
    print(f"[LOAD] Model loaded on {model.device}")
    return model, tokenizer

# Muat model (jalankan sekali)
model, tokenizer = load_model()

# -----------------------------------------------------------------------------
# 2. Data Classes
# -----------------------------------------------------------------------------
@dataclass
class LocationInfo:
    display_name: str
    city: str
    country: str
    lat: float
    lon: float

@dataclass
class WeatherInfo:
    temperature: float
    humidity: float
    wind_speed: float
    wind_direction: float
    timestamp: str

@dataclass
class SunPathInfo:
    solar_declination: float
    optimal_orientation: str
    hemisphere: str
    solar_radiation: float
    sunrise: str
    sunset: str

@dataclass
class NoiseInfo:
    nearest_road_distance_km: float
    estimated_db: float

@dataclass
class EnvironmentReport:
    location: LocationInfo
    weather: WeatherInfo
    sun_path: SunPathInfo
    noise: NoiseInfo
    timestamp: str

# -----------------------------------------------------------------------------
# 3. Core Functions
# -----------------------------------------------------------------------------
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad)*math.cos(lat2_rad)*math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

def calculate_solar_declination(lat):
    now = datetime.now()
    day_of_year = now.timetuple().tm_yday
    declination = 23.44 * math.sin(math.radians((360/365)*(284+day_of_year)))
    if lat > 0:
        orientation = "North-facing"
        hemisphere = "Northern"
    elif lat < 0:
        orientation = "South-facing"
        hemisphere = "Southern"
    else:
        orientation = "East/West (equatorial)"
        hemisphere = "Equatorial"
    return {
        "solar_declination": round(declination, 2),
        "optimal_orientation": orientation,
        "hemisphere": hemisphere
    }

# -----------------------------------------------------------------------------
# 4. API Calls
# -----------------------------------------------------------------------------
@lru_cache(maxsize=128)
def reverse_geocode(lat: float, lon: float) -> LocationInfo:
    time.sleep(1)
    url = "https://nominatim.openstreetmap.org/reverse"
    params = {
        "lat": lat,
        "lon": lon,
        "format": "json",
        "zoom": 16,                     # lebih detail
        "addressdetails": 1,
        "accept-language": "id"
    }
    headers = {"User-Agent": "FluxAI-Environment/1.0"}
    try:
        response = requests.get(url, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()
        if "address" in data:
            address = data["address"]
            # Prioritas paling spesifik
            city = (
                address.get("neighbourhood") or
                address.get("suburb") or
                address.get("village") or
                address.get("town") or
                address.get("city") or
                address.get("county") or
                "Unknown"
            )
            country = address.get("country", "Unknown")
            return LocationInfo(
                display_name=data.get("display_name", ""),
                city=city,
                country=country,
                lat=lat,
                lon=lon
            )
        else:
            return LocationInfo("Unknown", "Unknown", "Unknown", lat, lon)
    except Exception as e:
        print(f"[REVERSE_GEOCODE] Error: {e}")
        return LocationInfo("Unknown (fallback)", "Unknown", "Unknown", lat, lon)

@lru_cache(maxsize=128)
def get_weather(lat, lon):
    time.sleep(0.5)
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m",
        "timezone": "auto"
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        if "current" in data:
            cur = data["current"]
            return WeatherInfo(
                cur.get("temperature_2m", 25.0),
                cur.get("relative_humidity_2m", 60.0),
                cur.get("wind_speed_10m", 3.0),
                cur.get("wind_direction_10m", 180.0),
                cur.get("time", datetime.now().isoformat())
            )
        else:
            return WeatherInfo(25.0, 60.0, 3.0, 180.0, datetime.now().isoformat())
    except Exception as e:
        print(f"[WEATHER] Error: {e}")
        return WeatherInfo(25.0, 60.0, 3.0, 180.0, datetime.now().isoformat())

@lru_cache(maxsize=128)
def get_solar_radiation(lat, lon):
    time.sleep(0.5)
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "sunrise,sunset",
        "current": "shortwave_radiation",
        "timezone": "auto",
        "forecast_days": 1
    }
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        ghi = data.get("current", {}).get("shortwave_radiation", 0.0)
        sunrise = data.get("daily", {}).get("sunrise", [""])[0] if data.get("daily", {}).get("sunrise") else ""
        sunset = data.get("daily", {}).get("sunset", [""])[0] if data.get("daily", {}).get("sunset") else ""
        return {"ghi": ghi, "sunrise": sunrise, "sunset": sunset}
    except Exception as e:
        print(f"[SOLAR_RADIATION] Error: {e}")
        return {"ghi": 0.0, "sunrise": "", "sunset": ""}

def query_overpass(lat, lon, endpoint, timeout=20):
    radius = 500
    query = f"""
    [out:json];
    way["highway"~"^(primary|secondary|tertiary|trunk|motorway)"](around:{radius},{lat},{lon});
    out center;
    """
    headers = {
        "Accept": "application/json",
        "User-Agent": "FluxAI-Environment/1.0",
        "Content-Type": "application/x-www-form-urlencoded"
    }
    try:
        response = requests.post(
            endpoint,
            data={"data": query},
            headers=headers,
            timeout=timeout
        )
        if response.status_code == 200:
            return response.json()
        else:
            print(f"[OVERPASS] {endpoint} status {response.status_code}")
            return None
    except Exception as e:
        print(f"[OVERPASS] {endpoint} error: {e}")
        return None

def estimate_noise(lat, lon):
    endpoints = [
        "https://overpass-api.de/api/interpreter",
        "https://overpass.kumi.systems/api/interpreter",
        "https://overpass.openstreetmap.fr/api/interpreter",
    ]
    data = None
    for ep in endpoints:
        print(f"[NOISE] Trying {ep} ...")
        data = query_overpass(lat, lon, ep, timeout=20)
        if data is not None:
            print(f"[NOISE] Success from {ep}")
            break
    if data is None:
        print("[NOISE] All endpoints failed, using fallback 45 dB")
        return NoiseInfo(nearest_road_distance_km=0.5, estimated_db=45.0)

    min_dist_km = float('inf')
    if "elements" in data:
        for elem in data["elements"]:
            if "center" in elem:
                c = elem["center"]
                rlat = c.get("lat")
                rlon = c.get("lon")
            elif "lat" in elem and "lon" in elem:
                rlat = elem["lat"]
                rlon = elem["lon"]
            else:
                continue
            if rlat is not None and rlon is not None:
                d = haversine_distance(lat, lon, rlat, rlon)
                if d < min_dist_km:
                    min_dist_km = d

    if min_dist_km == float('inf'):
        return NoiseInfo(nearest_road_distance_km=5.0, estimated_db=25.0)

    dist_m = min_dist_km * 1000
    if dist_m < 10:
        db = 70.0
    else:
        db = max(25.0, 70.0 - 20 * math.log10(dist_m + 1))

    return NoiseInfo(
        nearest_road_distance_km=round(min_dist_km, 4),
        estimated_db=round(db, 1)
    )

def analyze_environment(lat, lon):
    location = reverse_geocode(lat, lon)
    weather = get_weather(lat, lon)
    sun_calc = calculate_solar_declination(lat)
    solar_data = get_solar_radiation(lat, lon)
    noise = estimate_noise(lat, lon)

    return EnvironmentReport(
        location=location,
        weather=weather,
        sun_path=SunPathInfo(
            solar_declination=sun_calc["solar_declination"],
            optimal_orientation=sun_calc["optimal_orientation"],
            hemisphere=sun_calc["hemisphere"],
            solar_radiation=solar_data.get("ghi", 0.0),
            sunrise=solar_data.get("sunrise", ""),
            sunset=solar_data.get("sunset", "")
        ),
        noise=noise,
        timestamp=datetime.now().isoformat()
    )

# -----------------------------------------------------------------------------
# 5. LLM-Based Suggestion Generation
# -----------------------------------------------------------------------------
def generate_llm_suggestions(model, tokenizer, plan, env, scores):
    prompt = f"""You are a professional architectural consultant specializing in sustainable and climate-responsive design.

Evaluate the following building plan based on environmental data and provide specific, actionable recommendations.

ENVIRONMENTAL DATA:
- Location: {env.location.city}, {env.location.country} (lat {env.location.lat}, lon {env.location.lon})
- Current Temperature: {env.weather.temperature} C
- Humidity: {env.weather.humidity}%
- Wind Speed: {env.weather.wind_speed} m/s, direction {env.weather.wind_direction} deg from North
- Solar Declination: {env.sun_path.solar_declination} deg
- Optimal Building Orientation: {env.sun_path.optimal_orientation}
- Solar Radiation: {env.sun_path.solar_radiation} W/m2
- Noise Level: {env.noise.estimated_db} dB
- Distance to Nearest Major Road: {env.noise.nearest_road_distance_km} km

PLAN DATA:
- Name: {plan.get('name', 'Unnamed')}
- Orientation: {plan.get('orientation', 0)} deg from North
- Rooms: {', '.join([r.get('name') for r in plan.get('rooms', [])])}

ENVIRONMENTAL SCORES (0=worst, 1=best):
- Noise Score: {scores.get('noise_score', 0):.2f}
- Daylight Score: {scores.get('daylight_score', 0):.2f}
- Ventilation Score: {scores.get('ventilation_score', 0):.2f}
- Overall Score: {scores.get('env_score', 0):.2f}

Based on the data, write a professional assessment and recommendations.

Output format:
ASSESSMENT: (2-3 sentences summarizing the plan's environmental performance)
RECOMMENDATIONS:
- (specific recommendation 1 with technical details)
- (specific recommendation 2)
- (specific recommendation 3)
- (more if needed)

Do not use emojis, symbols, or informal language. Be concise but thorough.
"""
    messages = [
        {"role": "system", "content": "You are a helpful architectural consultant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.5,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    input_len = inputs.input_ids.shape[1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    if response.startswith("assistant"):
        response = response[len("assistant"):].strip()
    return response

# -----------------------------------------------------------------------------
# 6. Evaluation Function
# -----------------------------------------------------------------------------
def evaluate_plan_environment(model, tokenizer, plan, env, weights=None):
    if weights is None:
        weights = {"noise": 0.4, "daylight": 0.3, "ventilation": 0.3}

    orientation = plan.get('orientation', 0)
    rooms = plan.get('rooms', [])

    # Noise
    distance_km = env.noise.nearest_road_distance_km
    noise_raw = min(1.0, distance_km / 2.0)
    noise_score = noise_raw

    # Daylight
    optimal_text = env.sun_path.optimal_orientation.lower()
    if "south" in optimal_text:
        optimal_deg = 180
    elif "north" in optimal_text:
        optimal_deg = 0
    elif "east" in optimal_text:
        optimal_deg = 90
    elif "west" in optimal_text:
        optimal_deg = 270
    else:
        optimal_deg = 180

    diff = abs(orientation - optimal_deg) % 360
    diff = min(diff, 360 - diff)
    daylight_score = max(0, 1 - diff / 180)
    if env.sun_path.solar_radiation > 200:
        daylight_score = min(1.0, daylight_score * 1.2)

    # Ventilation
    wind_dir = env.weather.wind_direction
    vent_diff = abs(orientation - wind_dir) % 360
    vent_diff = min(vent_diff, 360 - vent_diff)
    vent_score = max(0, 1 - vent_diff / 180)

    window_orientations = set()
    for room in rooms:
        for win in room.get('windows', []):
            if 'direction' in win:
                window_orientations.add(win['direction'] % 360)
    if len(window_orientations) >= 2:
        orientations = list(window_orientations)
        for i in range(len(orientations)):
            for j in range(i+1, len(orientations)):
                if abs((orientations[i] - orientations[j]) % 360 - 180) < 30:
                    vent_score = min(1.0, vent_score * 1.3)
                    break

    env_score = (
        weights["noise"] * noise_score +
        weights["daylight"] * daylight_score +
        weights["ventilation"] * vent_score
    )

    scores_detail = {
        "noise_score": round(noise_score, 3),
        "daylight_score": round(daylight_score, 3),
        "ventilation_score": round(vent_score, 3),
        "env_score": round(env_score, 3)
    }

    mitigation = generate_llm_suggestions(model, tokenizer, plan, env, scores_detail)
    return env_score, mitigation, scores_detail

def rank_plans_by_environment(model, tokenizer, plans, env_report, initial_scores=None, weight_env=0.3):
    if initial_scores is None:
        initial_scores = [0.5] * len(plans)

    results = []
    for i, plan in enumerate(plans):
        env_score, mitigation, scores_detail = evaluate_plan_environment(model, tokenizer, plan, env_report)
        combined = (1 - weight_env) * initial_scores[i] + weight_env * env_score
        results.append({
            "plan": plan,
            "initial_score": initial_scores[i],
            "env_score": env_score,
            "combined_score": combined,
            "mitigation": mitigation,
            "scores_detail": scores_detail
        })

    results.sort(key=lambda x: x["combined_score"], reverse=True)
    return results

def run_evaluation(model, tokenizer, lat, lon, plans, initial_scores=None):
    print("="*70)
    print("ENVIRONMENT ANALYSIS AND PLAN EVALUATION")
    print(f"Location: lat={lat}, lon={lon}")
    print("="*70)

    env_report = analyze_environment(lat, lon)
    print("\n[ENV] Environment Report:")
    print(f"  City: {env_report.location.city}, {env_report.location.country}")
    print(f"  Temperature: {env_report.weather.temperature} C")
    print(f"  Humidity: {env_report.weather.humidity}%")
    print(f"  Wind: {env_report.weather.wind_speed} m/s, direction {env_report.weather.wind_direction} deg")
    print(f"  Solar declination: {env_report.sun_path.solar_declination} deg")
    print(f"  Optimal orientation: {env_report.sun_path.optimal_orientation}")
    print(f"  Solar radiation: {env_report.sun_path.solar_radiation} W/m2")
    print(f"  Sunrise: {env_report.sun_path.sunrise}, Sunset: {env_report.sun_path.sunset}")
    print(f"  Noise: {env_report.noise.estimated_db} dB (nearest road distance: {env_report.noise.nearest_road_distance_km} km)")

    if initial_scores is None:
        initial_scores = [0.5] * len(plans)

    ranked = rank_plans_by_environment(model, tokenizer, plans, env_report, initial_scores, weight_env=0.3)

    print("\n" + "="*70)
    print("FINAL RANKING")
    print("="*70)
    for idx, item in enumerate(ranked, 1):
        plan_name = item['plan'].get('name', f'Plan {idx}')
        print(f"{idx}. {plan_name}")
        print(f"   Combined: {item['combined_score']:.3f} (Initial: {item['initial_score']:.3f}, Environment: {item['env_score']:.3f})")
        mit_summary = item['mitigation'][:250] + "..." if len(item['mitigation']) > 250 else item['mitigation']
        print(f"   Mitigation: {mit_summary}")

    print("\n[DONE] Evaluation completed.")
    return ranked


import nest_asyncio
import uvicorn
import threading
import time
import subprocess
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

nest_asyncio.apply()

app = FastAPI(title="Environment Evaluator API")

class Window(BaseModel):
    direction: float

class Room(BaseModel):
    name: str
    center: List[float]
    windows: List[Window]

class Plan(BaseModel):
    name: str
    orientation: float
    rooms: List[Room]

class EvaluationRequest(BaseModel):
    lat: float
    lon: float
    plans: List[Plan]
    initial_scores: Optional[List[float]] = None

@app.post("/evaluate")
def evaluate_endpoint(req: EvaluationRequest):
    try:
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        print(f"\n[REQUEST][{timestamp}] Received evaluation request")
        print(f"  - Coordinates: lat={req.lat}, lon={req.lon}")
        print(f"  - Number of plans: {len(req.plans)}")
        if req.initial_scores is not None:
            print(f"  - Initial scores: {req.initial_scores}")
        for i, plan in enumerate(req.plans, 1):
            print(f"  - Plan {i}: '{plan.name}' orientation={plan.orientation} rooms={len(plan.rooms)}")

        plans_dict = [p.dict() for p in req.plans]
        initial_scores = req.initial_scores if req.initial_scores is not None else [0.5] * len(plans_dict)
        ranked = run_evaluation(model, tokenizer, req.lat, req.lon, plans_dict, initial_scores)

        print(f"[RESPONSE][{timestamp}] Evaluation completed successfully")
        print(f"  - Number of results: {len(ranked)}")
        for item in ranked:
            print(f"    -> {item['plan']['name']} | combined={item['combined_score']:.3f}")

        return {"status": "success", "results": ranked}
    except Exception as e:
        print(f"[ERROR][{time.strftime('%Y-%m-%d %H:%M:%S', time.localtime())}] Evaluation failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))

# --- Baca secret ngrok ---
try:
    user_secrets = UserSecretsClient()
    ngrok_auth_token = user_secrets.get_secret("NGROK_AUTH")
    print("[SUCCESS] Secret NGROK_AUTH berhasil dibaca.")
except Exception as e:
    print(f"[FAILED] Gagal membaca secret: {e}")
    raise

# --- Matikan proses di port 8000 dan 8001 ---
def kill_port(port):
    try:
        subprocess.run(["fuser", "-k", f"{port}/tcp"], check=False,
                       stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        time.sleep(1)
    except FileNotFoundError:
        subprocess.run(["bash", "-c", f"kill -9 $(lsof -t -i:{port}) 2>/dev/null"],
                       check=False, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
        time.sleep(1)

kill_port(8000)
kill_port(8001)

# Gunakan port 8001
PORT = 8001

# Set autentikasi ngrok
ngrok.set_auth_token(ngrok_auth_token)

# Mulai tunnel ngrok ke port 8001
public_url = ngrok.connect(PORT)
print(f"[CONNECTED] Ngrok tunnel tersedia di: {public_url}")

# Jalankan server FastAPI di thread terpisah
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

threading.Thread(target=run_server, daemon=True).start()

[LOAD] Loading Qwen/Qwen2.5-14B-Instruct with 4-bit...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[LOAD] Model loaded on cuda:0
[SUCCESS] Secret NGROK_AUTH berhasil dibaca.
[CONNECTED] Ngrok tunnel tersedia di: NgrokTunnel: "https://reshoot-levers-fraction.ngrok-free.dev" -> "http://localhost:8001"


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)



[REQUEST][2026-08-30 05:36:20] Received evaluation request
  - Coordinates: lat=-6.043664360294216, lon=106.6502437532297
  - Number of plans: 5
  - Initial scores: [0.75, 0.75, 0.7, 0.7, 0.7]
  - Plan 1: 'Variant 1 (seed 1085)' orientation=5.0 rooms=7
  - Plan 2: 'Variant 2 (seed 1119)' orientation=39.0 rooms=7
  - Plan 3: 'Variant 3 (seed 1051)' orientation=331.0 rooms=7
  - Plan 4: 'Variant 4 (seed 1068)' orientation=348.0 rooms=7
  - Plan 5: 'Variant 5 (seed 1102)' orientation=22.0 rooms=7
ENVIRONMENT ANALYSIS AND PLAN EVALUATION
Location: lat=-6.043664360294216, lon=106.6502437532297
[NOISE] Trying https://overpass-api.de/api/interpreter ...
[NOISE] Success from https://overpass-api.de/api/interpreter

[ENV] Environment Report:
  City: Pantai Indah Kapuk 2, Indonesia
  Temperature: 31.0 C
  Humidity: 56%
  Wind: 16.7 m/s, direction 27 deg
  Solar declination: 8.48 deg
  Optimal orientation: South-facing
  Solar radiation: 894.0 W/m2
  Sunrise: 2026-08-30T05:54, Sunset: 2026-08-30